# Geradores de código de modelos

Como mencionado antes no curso, existem ferramentas que geram modelos Pydantic para nós. Uma delas é o `datamodel-code-generator`, principalmente uma ferramenta de linha de comando.

Normalmente o gerador roda **uma vez**: o código gerado é "colado" na aplicação e não o executamos de novo a cada vez (claro, depende do que a aplicação faz).

A documentação do gerador está [aqui](https://github.com/koxudaxi/datamodel-code-generator/tree/main).

Nesta pasta há alguns arquivos adicionais:
- um JSON Schema simples ([`schema_1.json`](schema_1.json));
- um JSON Schema mais complicado ([`schema_2.json`](schema_2.json));
- um JSON Schema ainda mais complicado ([`schema_3.json`](schema_3.json));
- um arquivo com dados JSON de exemplo (sem schema, só dados) ([`rectangle_data.json`](rectangle_data.json));
- um arquivo com dados CSV ([`csv_data.csv`](csv_data.csv)).

> 📝 **Nota de revisão — reexecução:** as saídas do gerador mostradas no curso são de novembro de 2023. Neste notebook, cada exemplo mantém o código do curso **e** reexecuta o gerador instalado (`pip install datamodel-code-generator`), com os mesmos parâmetros mais `--disable-timestamp` (para a saída não mudar a cada execução). Diferenças encontradas na versão atual: `Optional[X]` virou `X | None` e `List[X]` virou `list[X]` (por causa de `--target-python-version 3.12`); o resto é igual.

In [1]:
import shutil
import subprocess
import sys

CODEGEN_ARGS = [
    "--output-model-type", "pydantic_v2.BaseModel", "--target-python-version", "3.12",
    "--allow-population-by-field-name", "--field-constraints", "--snake-case-field", "--disable-timestamp",
]


def codegen(input_file: str, file_type: str, *extra: str) -> str:
    """Roda o datamodel-codegen e devolve o código gerado (ou um aviso, se não estiver instalado)."""
    exe = shutil.which("datamodel-codegen")
    if exe is None:
        return "# datamodel-codegen não instalado: pip install datamodel-code-generator"
    result = subprocess.run([exe, "--input", input_file, "--input-file-type", file_type, *CODEGEN_ARGS, *extra],
                            capture_output=True, text=True, check=True)
    return result.stdout.strip()


print(f"versão do gerador: {subprocess.run([shutil.which('datamodel-codegen') or 'true', '--version'], capture_output=True, text=True).stdout.strip()}")

versão do gerador: datamodel-codegen 0.93.0


## JSON Schema — exemplo 1

O arquivo `schema_1.json` é um schema bem simples:

```JSON
{
  "$id": "https://example.com/person.schema.json",
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "Person",
  "type": "object",
  "properties": {
    "firstName": {
      "type": "string",
      "description": "The person's first name."
    },
    "lastName": {
      "type": "string",
      "description": "The person's last name."
    },
    "age": {
      "description": "Age in years which must be equal to or greater than zero.",
      "type": "integer",
      "minimum": 0
    }
  }
}
```

Para gerar um modelo Pydantic a partir dele, usamos esta linha de comando:
```sh
datamodel-codegen --input schema_1.json --input-file-type jsonschema --output-model-type pydantic_v2.BaseModel --target-python-version "3.12" --allow-population-by-field-name --field-constraints --snake-case-field
```

A saída (no curso, em 2023):

In [2]:
# generated by datamodel-codegen:
#   filename:  schema_1.json
#   timestamp: 2023-11-21T04:54:05+00:00

from __future__ import annotations

from typing import Optional

from pydantic import BaseModel, ConfigDict, Field


class Person(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    first_name: Optional[str] = Field(
        None, alias='firstName', description="The person's first name."
    )
    last_name: Optional[str] = Field(
        None, alias='lastName', description="The person's last name."
    )
    age: Optional[int] = Field(
        None,
        description='Age in years which must be equal to or greater than zero.',
        ge=0,
    )

Reexecutando hoje (saída do gerador **atual**, mesmos parâmetros):

In [3]:
print(f'codegen("schema_1.json", "jsonschema"):\n{codegen("schema_1.json", "jsonschema")}')

codegen("schema_1.json", "jsonschema"):
# generated by datamodel-codegen:
#   filename:  schema_1.json

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class Person(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    first_name: str | None = Field(
        None, alias='firstName', description="The person's first name."
    )
    last_name: str | None = Field(
        None, alias='lastName', description="The person's last name."
    )
    age: int | None = Field(
        None,
        description='Age in years which must be equal to or greater than zero.',
        ge=0,
    )


O import de `annotations` do `__future__` não é necessário neste caso (tem a ver com anotações de tipo que fazem referências "adiantadas" ou adiadas — mais informações [aqui](https://peps.python.org/pep-0563/)). Mas importá-lo não faz mal.

> 📝 **Nota de revisão:** no original, este parágrafo estava numa célula de **código** (texto solto, que daria `SyntaxError` ao executar). Convertido para markdown.

O resultado é muito parecido com os modelos Pydantic que viemos escrevendo.

Na verdade, converter de JSON Schema (ou OpenAPI) para modelos Pydantic funciona bem, em geral, para schemas não muito complicados.

In [4]:
p = Person.model_validate({"firstName": "Ada", "lastName": "Lovelace", "age": 36})
print(f"p: {p!r}")
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)}")

p: Person(first_name='Ada', last_name='Lovelace', age=36)
p.model_dump(by_alias=True): {'firstName': 'Ada', 'lastName': 'Lovelace', 'age': 36}


## JSON Schema — exemplo 2

Neste segundo exemplo, o JSON é um pouco mais complicado:

```JSON
{
  "$id": "https://example.com/complex-object.schema.json",
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "Complex Object",
  "type": "object",
  "properties": {
    "name": {
      "type": "string"
    },
    "age": {
      "type": "integer",
      "minimum": 0
    },
    "address": {
      "type": "object",
      "properties": {
        "street": {
          "type": "string"
        },
        "city": {
          "type": "string"
        },
        "state": {
          "type": "string"
        },
        "postalCode": {
          "type": "string",
          "pattern": "\\d{5}"
        }
      },
      "required": [
        "street",
        "city",
        "state",
        "postalCode"
      ]
    },
    "hobbies": {
      "type": "array",
      "items": {
        "type": "string"
      }
    }
  },
  "required": [
    "name",
    "age"
  ]
}
```

Rodando o gerador:

```shell
datamodel-codegen --input schema_2.json --input-file-type jsonschema --output-model-type pydantic_v2.BaseModel --target-python-version "3.12" --allow-population-by-field-name --field-constraints --snake-case-field
```

E esta é a saída (no curso):

In [5]:
# generated by datamodel-codegen:
#   filename:  schema_2.json
#   timestamp: 2023-11-21T04:56:59+00:00

from __future__ import annotations

from typing import List, Optional

from pydantic import BaseModel, ConfigDict, Field


class Address(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    street: str
    city: str
    state: str
    postal_code: str = Field(..., alias='postalCode', pattern='\\d{5}')


class ComplexObject(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    name: str
    age: int = Field(..., ge=0)
    address: Optional[Address] = None
    hobbies: Optional[List[str]] = None

Reexecutando hoje (saída do gerador **atual**, mesmos parâmetros):

In [6]:
print(f'codegen("schema_2.json", "jsonschema"):\n{codegen("schema_2.json", "jsonschema")}')

codegen("schema_2.json", "jsonschema"):
# generated by datamodel-codegen:
#   filename:  schema_2.json

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class Address(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    street: str
    city: str
    state: str
    postal_code: str = Field(..., alias='postalCode', pattern='\\d{5}')


class ComplexObject(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    name: str
    age: int = Field(..., ge=0)
    address: Address | None = None
    hobbies: list[str] | None = None


De novo, o gerador funcionou muito bem.

Aliás, se você está se perguntando o que aquele `...` (*ellipsis*) faz nos `Field`: ele só indica que o campo não tem default. É uma herança do Pydantic V1, em que campos opcionais recebiam automaticamente o default `None` a não ser que se passasse a ellipsis. Isso não acontece mais no Pydantic V2, como vimos no curso, então não precisamos mantê-la:

In [7]:
class Address(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    street: str
    city: str
    state: str
    postal_code: str = Field(alias='postalCode', pattern='\\d{5}')


class ComplexObject(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    name: str
    age: int = Field(ge=0)
    address: Optional[Address] = None
    hobbies: Optional[List[str]] = None

> 📝 **Nota de revisão — `pattern` não é ancorado:** em JSON Schema, `pattern` é uma **busca** (casa em qualquer parte da string), e o Pydantic segue a mesma semântica. Então `\d{5}` aceita `"123456"` e até `"abc12345xyz"` como CEP. O gerador traduziu fielmente — o problema está no schema, que provavelmente queria `^\d{5}$`. Vale revisar todo `pattern` gerado:

In [8]:
from pydantic import ValidationError

base = {"street": "Rua A", "city": "Recife", "state": "PE"}
for postal_code in ["12345", "123456", "abc12345xyz", "1234"]:
    try:
        Address.model_validate(base | {"postalCode": postal_code})
        print(f"{postal_code!r}: aceito")
    except ValidationError as ex:
        print(f"{postal_code!r}: rejeitado ({ex.errors()[0]['type']})")

class AnchoredAddress(Address):
    postal_code: str = Field(alias="postalCode", pattern=r"^\d{5}$")

for postal_code in ["12345", "123456", "abc12345xyz"]:
    try:
        AnchoredAddress.model_validate(base | {"postalCode": postal_code})
        print(f"ancorado {postal_code!r}: aceito")
    except ValidationError as ex:
        print(f"ancorado {postal_code!r}: rejeitado ({ex.errors()[0]['type']})")

'12345': aceito
'123456': aceito
'abc12345xyz': aceito
'1234': rejeitado (string_pattern_mismatch)
ancorado '12345': aceito
ancorado '123456': rejeitado (string_pattern_mismatch)
ancorado 'abc12345xyz': rejeitado (string_pattern_mismatch)


## JSON Schema — exemplo 3

Mas, mesmo com um JSON Schema, há coisas que o gerador simplesmente não consegue tratar.

Neste terceiro exemplo (`schema_3.json`), temos este JSON Schema:

```json
{
  "$id": "https://example.com/conditional-validation-if-else.schema.json",
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "title": "Conditional Validation with If-Else",
  "type": "object",
  "properties": {
    "isMember": {
      "type": "boolean"
    },
    "membershipNumber": {
      "type": "string"
    }
  },
  "required": [
    "isMember"
  ],
  "if": {
    "properties": {
      "isMember": {
        "const": true
      }
    }
  },
  "then": {
    "properties": {
      "membershipNumber": {
        "type": "string",
        "minLength": 10,
        "maxLength": 10
      }
    }
  },
  "else": {
    "properties": {
      "membershipNumber": {
        "type": "string",
        "minLength": 5
      }
    }
  }
}
```

> 📝 **Nota de revisão:** este trecho também estava numa célula de código no original. Convertido para markdown.

Segundo o schema, há dois campos: `isMember` (obrigatório, booleano) e `membershipNumber` (opcional, string).

Porém, há uma **condicional**: se `isMember` for `True`, `membershipNumber` precisa ter exatamente `10` caracteres; caso contrário, pelo menos `5`.

Rodando o gerador:

```shell
datamodel-codegen --input schema_3.json --input-file-type jsonschema --output-model-type pydantic_v2.BaseModel --target-python-version "3.12" --allow-population-by-field-name --field-constraints --snake-case-field
```

Obtemos este modelo (no curso):

In [9]:
# generated by datamodel-codegen:
#   filename:  schema_3.json
#   timestamp: 2023-11-21T05:32:20+00:00

from __future__ import annotations

from typing import Optional

from pydantic import BaseModel, ConfigDict, Field


class ConditionalValidationWithIfElse(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    is_member: bool = Field(..., alias='isMember')
    membership_number: Optional[str] = Field(None, alias='membershipNumber')

Reexecutando hoje (saída do gerador **atual**, mesmos parâmetros):

In [10]:
print(f'codegen("schema_3.json", "jsonschema"):\n{codegen("schema_3.json", "jsonschema")}')

codegen("schema_3.json", "jsonschema"):
# generated by datamodel-codegen:
#   filename:  schema_3.json

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class ConditionalValidationWithIfElse(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    is_member: bool = Field(..., alias='isMember')
    membership_number: str | None = Field(None, alias='membershipNumber')


O que obviamente não captura tudo o que o schema define.

Conseguimos fazer isso nós mesmos? Como?

Teríamos que usar validadores customizados para tratar as condições impostas pelo schema dependente.

A esta altura você já deve conseguir fazer isso sozinho, mas vamos juntos de qualquer forma.

Olhando a condicional do schema, na prática `membership_number` **não** é opcional e não pode ser `None` em nenhum caso. Então corrigimos isso primeiro (também dispensamos a ellipsis, e escolhemos um nome de modelo mais curto):

> 📝 **Nota de revisão:** "não pode ser `None`" é uma leitura possível, mas não a única: pelo schema, `membershipNumber` não está em `required`, então um objeto **sem** o campo é válido em JSON Schema (as condições só restringem o campo quando ele existe). O instrutor escolheu tornar o campo obrigatório — é uma decisão de negócio razoável, mas é mais restritiva que o schema.

In [11]:
class MemberInfo(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    is_member: bool = Field(alias='isMember')
    membership_number: str = Field(alias='membershipNumber')

Em seguida, um validador customizado que verifica se `membership_number` tem exatamente 10 caracteres quando `is_member` é `True`, ou pelo menos `5` caso contrário. Ou seja, em qualquer caso o número precisa ter pelo menos `5` caracteres — colocamos isso na definição do campo e só checamos os `10` caracteres quando `is_member` for `True`:

> 📝 **Nota de revisão:** o validador abaixo não tem `@classmethod` nem `cls`. O Pydantic aceita (ele inspeciona a assinatura), mas a convenção documentada é `@classmethod` + `cls` — é o que usamos no resto do material.

In [12]:
from pydantic import field_validator, ValidationInfo


class MemberInfo(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    is_member: bool = Field(alias='isMember')
    membership_number: str = Field(alias='membershipNumber', min_length=5)

    @field_validator("membership_number")  # after validator
    def validate_membership_number(value: str, validation_info: ValidationInfo):
        print("validating membership_number...")
        values = validation_info.data
        if "is_member" in values:
            # is_member validated OK, so we have a bool value for it
            is_member = values["is_member"]
            if is_member:
                if len(value) != 10:
                    raise ValueError("must be exactly 10 characters long when is_member is set to True")
        # is_member failed validation, not much we can do, so just return value
        return value

Testando o modelo:

In [13]:
print(f'MemberInfo(is_member=False, membership_number="12345"): {MemberInfo(is_member=False, membership_number="12345")!r}')

validating membership_number...
MemberInfo(is_member=False, membership_number="12345"): MemberInfo(is_member=False, membership_number='12345')


In [14]:
print(f'MemberInfo(is_member=True, membership_number = "0123456789"): {MemberInfo(is_member=True, membership_number = "0123456789")!r}')

validating membership_number...
MemberInfo(is_member=True, membership_number = "0123456789"): MemberInfo(is_member=True, membership_number='0123456789')


In [15]:
from pydantic import ValidationError

In [16]:
try:
    MemberInfo(is_member=False)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for MemberInfo
membershipNumber
  Field required [type=missing, input_value={'is_member': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [17]:
try:
    MemberInfo(is_member=False, membershipNumber="123")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 1 validation error for MemberInfo
membershipNumber
  String should have at least 5 characters [type=string_too_short, input_value='123', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_short


In [18]:
try:
    MemberInfo(is_member=True, membershipNumber="123456789012345")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

validating membership_number...
ValidationError: 1 validation error for MemberInfo
membershipNumber
  Value error, must be exactly 10 characters long when is_member is set to True [type=value_error, input_value='123456789012345', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


> 💡 Repare que, quando `membershipNumber` falta ou é curto demais, a mensagem "validating membership_number..." **não** aparece: a restrição `min_length=5` do campo falha antes, e o validador *after* nem chega a rodar.

## Gerando um modelo a partir de dados JSON reais

Às vezes (na verdade, mais do que gostaríamos), quem desenvolve a API não se dá ao trabalho de especificar um JSON Schema para os modelos.

(Por quê? O instrutor não sabe: preguiça, desinformação, descaso… A opinião dele: se você cria uma API REST, especifica JSON Schemas para todo JSON que trafega nela.)

O que fazer nesses casos? Podemos fazer como antes no curso: inspecionar o JSON visualmente e montar um modelo adequado.

É assim que o instrutor faria. Vejamos por que ele não gosta do gerador neste caso (não é culpa do gerador, que faz o melhor possível; às vezes há sutilezas nos dados que nós, humanos, deduzimos e o gerador não — pelo menos até nossos senhores de IA assumirem!)

Nesta pasta há um arquivo JSON (`rectangle_data.json`) com estes dados:

```json
{
    "rectangles": [
        {
            "upperLeft": {"x": 10, "y": 20},
            "lowerRight": {"x": 100, "y": 50}
        },
        {
            "upperLeft": {"x": 10.5, "y": 20.5},
            "lowerRight": {"x": 100.5, "y": 50.5}
        }
    ]
}
```

E esta é a saída depois de rodar o gerador (no curso):
```shell
datamodel-codegen --input rectangle_data.json --input-file-type json --output-model-type pydantic_v2.BaseModel --target-python-version "3.12" --allow-population-by-field-name --field-constraints --snake-case-field
```

In [19]:
# generated by datamodel-codegen:
#   filename:  rectangle_data.json
#   timestamp: 2023-11-21T05:48:25+00:00

from __future__ import annotations

from typing import List

from pydantic import BaseModel, ConfigDict, Field


class UpperLeft(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    x: float
    y: float


class LowerRight(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    x: float
    y: float


class Rectangle(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    upper_left: UpperLeft = Field(..., alias='upperLeft')
    lower_right: LowerRight = Field(..., alias='lowerRight')


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    rectangles: List[Rectangle]

Reexecutando hoje (saída do gerador **atual**, mesmos parâmetros):

In [20]:
print(f'codegen("rectangle_data.json", "json"):\n{codegen("rectangle_data.json", "json")}')

codegen("rectangle_data.json", "json"):
# generated by datamodel-codegen:
#   filename:  rectangle_data.json

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class UpperLeft(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    x: float
    y: float


class LowerRight(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    x: float
    y: float


class Rectangle(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    upper_left: UpperLeft = Field(..., alias='upperLeft')
    lower_right: LowerRight = Field(..., alias='lowerRight')


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    rectangles: list[Rectangle]


Os modelos não são ruins — na verdade, funcionariam bem.

Mas veja `UpperLeft` e `LowerRight`: são exatamente o mesmo modelo. Não precisamos de dois modelos separados; podemos referenciar um modelo comum nos campos `upper_left` e `lower_right` (e removo as ellipsis desnecessárias):

In [21]:
class Coordinates(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    x: float
    y: float


class Rectangle(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    upper_left: Coordinates = Field(alias='upperLeft')
    lower_right: Coordinates = Field(alias='lowerRight')


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    rectangles: List[Rectangle]

> 📝 **Nota de revisão — `--reuse-model`:** o gerador tem uma opção para isso (ela já aparece no `--help` listado no fim desta aula). Ela não chega a fundir os modelos num só, mas faz o segundo herdar do primeiro (`class LowerRight(UpperLeft): pass`), eliminando a duplicação:

In [22]:
print(f'codegen("rectangle_data.json", "json", "--reuse-model"):\n{codegen("rectangle_data.json", "json", "--reuse-model")}')

codegen("rectangle_data.json", "json", "--reuse-model"):
# generated by datamodel-codegen:
#   filename:  rectangle_data.json

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class UpperLeft(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    x: float
    y: float


class LowerRight(UpperLeft):
    pass


class Rectangle(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    upper_left: UpperLeft = Field(..., alias='upperLeft')
    lower_right: LowerRight = Field(..., alias='lowerRight')


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    rectangles: list[Rectangle]


In [23]:
import json
from pathlib import Path

rectangles = Model.model_validate_json(Path("rectangle_data.json").read_text())
print(f"rectangles.rectangles[1].lower_right: {rectangles.rectangles[1].lower_right!r}")

rectangles.rectangles[1].lower_right: Coordinates(x=100.5, y=50.5)


## Gerando um modelo a partir de um arquivo CSV

Numa aula anterior, criamos manualmente um modelo para o arquivo `pop_estimates.csv`.

Também dá para gerá-lo com o gerador, mas, como costuma acontecer, nenhum dos validadores especializados que escrevemos à mão (porque entendemos algo sobre os dados além dos próprios dados, algo que o gerador não consegue) será gerado, e o modelo fica bem simplista.

```shell
datamodel-codegen --input pop_estimates.csv --input-file-type csv --output-model-type pydantic_v2.BaseModel --target-python-version "3.12" --allow-population-by-field-name --field-constraints --snake-case-field
```

In [24]:
# generated by datamodel-codegen:
#   filename:  pop_estimates.csv
#   timestamp: 2023-11-21T04:32:37+00:00

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    geographic_area: str = Field(..., alias='Geographic Area')
    july_1__2001_estimate: str = Field(..., alias='July 1, 2001 Estimate')
    july_1__2000_estimate: str = Field(..., alias='July 1, 2000 Estimate')
    april_1__2000_population_estimates_base: str = Field(
        ..., alias='April 1, 2000 Population Estimates Base'
    )

Reexecutando hoje (saída do gerador **atual**, mesmos parâmetros):

In [25]:
print(f'codegen("pop_estimates.csv", "csv"):\n{codegen("pop_estimates.csv", "csv")}')

codegen("pop_estimates.csv", "csv"):
# generated by datamodel-codegen:
#   filename:  pop_estimates.csv

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    geographic_area: str = Field(..., alias='Geographic Area')
    july_1__2001_estimate: str = Field(..., alias='July 1, 2001 Estimate')
    july_1__2000_estimate: str = Field(..., alias='July 1, 2000 Estimate')
    april_1__2000_population_estimates_base: str = Field(
        ..., alias='April 1, 2000 Population Estimates Base'
    )


O gerador simplesmente define todas as colunas como string. O modelo está tecnicamente correto para o arquivo, mas não serve ao nosso propósito. Já vimos como criar um modelo mais adequado a esses dados.

E, para confirmar que o gerador não tenta inferir tipos a partir dos dados do CSV, usamos o arquivo `csv_data.csv`, com dados totalmente limpos de vários tipos.

Rodando o gerador:
```shell
datamodel-codegen --input csv_data.csv --input-file-type csv --output-model-type pydantic_v2.BaseModel --target-python-version "3.12" --allow-population-by-field-name --field-constraints --snake-case-field
```

obtemos este modelo:

In [26]:
# generated by datamodel-codegen:
#   filename:  csv_data.csv
#   timestamp: 2023-11-21T06:03:37+00:00

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    field_bool: str = Field(..., alias='fieldBool')
    field_str: str = Field(..., alias='fieldStr')
    field_int: str = Field(..., alias='fieldInt')
    field_float: str = Field(..., alias='fieldFloat')

Reexecutando hoje (saída do gerador **atual**, mesmos parâmetros):

In [27]:
print(f'codegen("csv_data.csv", "csv"):\n{codegen("csv_data.csv", "csv")}')

codegen("csv_data.csv", "csv"):
# generated by datamodel-codegen:
#   filename:  csv_data.csv

from __future__ import annotations

from pydantic import BaseModel, ConfigDict, Field


class Model(BaseModel):
    model_config = ConfigDict(
        populate_by_name=True,
    )
    field_bool: str = Field(..., alias='fieldBool')
    field_str: str = Field(..., alias='fieldStr')
    field_int: str = Field(..., alias='fieldInt')
    field_float: str = Field(..., alias='fieldFloat')


In [28]:
import csv

with open("csv_data.csv", newline="") as f:
    rows = [Model.model_validate(row) for row in csv.DictReader(f)]
print(f"rows[0]: {rows[0]!r}")

rows[0]: Model(field_bool='True', field_str='python', field_int='10', field_float='10.5')


> 💡 Todos os campos ficaram `str`: `'True'`, `'10'` e `'10.5'` continuam texto. Trocando as anotações à mão para `bool`, `int` e `float`, o próprio Pydantic faria a coerção (modo *lax*) — o gerador só não arrisca inferir.

## Conclusão

Como todo gerador de código, os resultados variam. O melhor conselho do instrutor, se você não quer escrever os modelos à mão: experimente e veja se o resultado é um ponto de partida bom o suficiente.

Ele, pessoalmente, usa pouco o gerador — com frequência demais acaba retrabalhando muito os modelos, e acha mais fácil começar do zero.

Há muitas opções disponíveis no gerador — só algumas foram vistas aqui.

O melhor jeito de descobrir as opções é rodar:
```shell
datamodel-codegen --help
```

que mostra (versão usada no curso, 2023):

> 📝 **Nota de revisão:** a versão atual tem bem mais opções (por exemplo `--formatters`, `--use-annotated`, `--class-name`, `--use-title-as-name`); rode o `--help` na sua instalação para a lista em vigor.

```
usage: 
  datamodel-codegen [options]

Generate Python data models from schema definitions or structured data

Options:
  --http-headers HTTP_HEADER [HTTP_HEADER ...]
                        Set headers in HTTP requests to the remote host. (example:
                        "Authorization: Basic dXNlcjpwYXNz")
  --http-ignore-tls     Disable verification of the remote host's TLS certificate
  --input INPUT         Input file/directory (default: stdin)
  --input-file-type {auto,openapi,jsonschema,json,yaml,dict,csv}
                        Input file type (default: auto)
  --output OUTPUT       Output file (default: stdout)
  --output-model-type {pydantic.BaseModel,pydantic_v2.BaseModel,dataclasses.dataclass,typing.TypedDict,msgspec.Struct}
                        Output model type (default: pydantic.BaseModel)
  --url URL             Input file URL. `--input` is ignored when `--url` is used

Typing customization:
  --base-class BASE_CLASS
                        Base Class (default: pydantic.BaseModel)
  --enum-field-as-literal {all,one}
                        Parse enum field as literal. all: all enum field type are Literal. one:
                        field type is Literal when an enum has only one possible value
  --field-constraints   Use field constraints and not con* annotations
  --set-default-enum-member
                        Set enum members as default values for enum field
  --strict-types {str,bytes,int,float,bool} [{str,bytes,int,float,bool} ...]
                        Use strict types
  --use-annotated       Use typing.Annotated for Field(). Also, `--field-constraints` option will
                        be enabled.
  --use-generic-container-types
                        Use generic container types for type hinting (typing.Sequence,
                        typing.Mapping). If `--use-standard-collections` option is set, then
                        import from collections.abc instead of typing
  --use-non-positive-negative-number-constrained-types
                        Use the Non{Positive,Negative}{FloatInt} types instead of the
                        corresponding con* constrained types.
  --use-one-literal-as-default
                        Use one literal as default value for one literal field
  --use-standard-collections
                        Use standard collections for type hinting (list, dict)
  --use-subclass-enum   Define Enum class as subclass with field type when enum has type (int,
                        float, bytes, str)
  --use-union-operator  Use | operator for Union type (PEP 604).
  --use-unique-items-as-set
                        define field type as `set` when the field attribute has `uniqueItems`

Field customization:
  --capitalise-enum-members, --capitalize-enum-members
                        Capitalize field names on enum
  --empty-enum-field-name EMPTY_ENUM_FIELD_NAME
                        Set field name when enum value is empty (default: `_`)
  --field-extra-keys FIELD_EXTRA_KEYS [FIELD_EXTRA_KEYS ...]
                        Add extra keys to field parameters
  --field-extra-keys-without-x-prefix FIELD_EXTRA_KEYS_WITHOUT_X_PREFIX [FIELD_EXTRA_KEYS_WITHOUT_X_PREFIX ...]
                        Add extra keys with `x-` prefix to field parameters. The extra keys are
                        stripped of the `x-` prefix.
  --field-include-all-keys
                        Add all keys to field parameters
  --force-optional      Force optional for required fields
  --original-field-name-delimiter ORIGINAL_FIELD_NAME_DELIMITER
                        Set delimiter to convert to snake case. This option only can be used with
                        --snake-case-field (default: `_` )
  --remove-special-field-name-prefix
                        Remove field name prefix if it has a special meaning e.g. underscores
  --snake-case-field    Change camel-case field name to snake-case
  --special-field-name-prefix SPECIAL_FIELD_NAME_PREFIX
                        Set field name prefix when first character can't be used as Python field
                        name (default: `field`)
  --strip-default-none  Strip default None on fields
  --use-default         Use default value even if a field is required
  --use-default-kwarg   Use `default=` instead of a positional argument for Fields that have
                        default values.
  --use-field-description
                        Use schema description to populate field docstring

Model customization:
  --allow-extra-fields  Allow to pass extra fields, if this flag is not passed, extra fields are
                        forbidden.
  --allow-population-by-field-name
                        Allow population by field name
  --class-name CLASS_NAME
                        Set class name of root model
  --collapse-root-models
                        Models generated with a root-type field will be mergedinto the models
                        using that root-type model
  --disable-appending-item-suffix
                        Disable appending `Item` suffix to model name in an array
  --disable-timestamp   Disable timestamp on file headers
  --enable-faux-immutability
                        Enable faux immutability
  --enable-version-header
                        Enable package version on file headers
  --keep-model-order    Keep generated models' order
  --reuse-model         Re-use models on the field when a module has the model with the same
                        content
  --target-python-version {3.6,3.7,3.8,3.9,3.10,3.11,3.12}
                        target python version (default: 3.7)
  --use-schema-description
                        Use schema description to populate class docstring
  --use-title-as-name   use titles as class names of models

Template customization:
  --aliases ALIASES     Alias mapping file
  --custom-file-header CUSTOM_FILE_HEADER
                        Custom file header
  --custom-file-header-path CUSTOM_FILE_HEADER_PATH
                        Custom file header file path
  --custom-template-dir CUSTOM_TEMPLATE_DIR
                        Custom template directory
  --encoding ENCODING   The encoding of input and output (default: UTF-8)
  --extra-template-data EXTRA_TEMPLATE_DATA
                        Extra template data
  --use-double-quotes   Model generated with double quotes. Single quotes or your black config
                        skip_string_normalization value will be used without this option.
  --wrap-string-literal
                        Wrap string literal by using black `experimental-string-processing`
                        option (require black 20.8b0 or later)

OpenAPI-only options:
  --openapi-scopes {schemas,paths,tags,parameters} [{schemas,paths,tags,parameters} ...]
                        Scopes of OpenAPI model generation (default: schemas)
  --strict-nullable     Treat default field as a non-nullable field (Only OpenAPI)
  --use-operation-id-as-name
                        use operation id of OpenAPI as class names of models
  --validation          Deprecated: Enable validation (Only OpenAPI). this option is deprecated.
                        it will be removed in future releases

General options:
  --debug               show debug message (require "debug". `$ pip install 'datamodel-code-
                        generator[debug]'`)
  --disable-warnings    disable warnings
  --no-color            disable colorized output
  --version             show version
  -h, --help            show this help message and exit
```